In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from IPython.display import display, Markdown

mpl.rcParams['font.size'] = 16

In [2]:
def read_file(filename):
    with open(filename) as f:
        data = []
        for line in f:
            line =  line.rstrip().lstrip()
            if not line:
                continue

            temp = line.split()
            data.append(float(temp[4]))

    return data

def read_file_with_smiles(filename):
    with open(filename) as f:
        data = []
        for line in f:
            line =  line.rstrip().lstrip()
            if not line:
                continue

            temp = line.split()
            data.append(float(temp[5]))

    return data

def plot_hist(data, output):
    fig = plt.figure(figsize=(10, 5))
    ax1 = fig.add_subplot(111)
    ax1.hist(data, bins=range(0, int(np.max(data))))
    ax1.set_xlabel('time [ms]')
    #ax1.set_xlim(0, 90)
    ax1.set_ylabel('Queries #')

    fig.savefig(f'{output}-hist.png', dpi=300)
    plt.show()

def percentiles(filename):
    data = read_file(filename)
    return np.percentile(data, 99), np.percentile(data, 90), np.percentile(data, 75), np.percentile(data, 50), np.percentile(data, 25)

def plots(filename, title, smiles=False):
    if smiles:
        data = read_file_with_smiles(filename)
    else:
        data = read_file(filename)
    md_output=f'''
## {title}
**Total time**: {np.sum(data):.3f} ms

**Average time**: {np.mean(data):.3f} ms

| Queries resolved (percentile)  | Time (ms) |
| -------------------------------| -----------|
| 99  | {np.percentile(data, 99):.3f} |
| 90  | {np.percentile(data, 90):.3f} |
| 75  | {np.percentile(data, 75):.3f} |
| 50  | {np.percentile(data, 50):.3f} |
| 25  | {np.percentile(data, 25):.3f} |
    '''
    display(Markdown(md_output))
    plot_hist(data, title)

In [4]:
percentile_data = {}
cutoffs =  {"10": 0.1, "11": 0.11, "12": 0.12,"13": 0.13, "14": 0.14, "15": 0.15, "16": 0.16,
           "17": 0.17, "18": 0.18, "19": 0.19, "20": 0.20, "21": 0.21, "22": 0.22, "23": 0.23, "24": 0.24, "25": 0.25 }

percentile_data["FPSim2"] = percentiles(f"../fpsim2_benchmark/benchmarking_b1024_0_7.log")
for key, value in cutoffs.items():
    percentile_data[str(value)] = percentiles(f"benchmark_{key}.log")

In [5]:
header = ["cut-off", "99%", "90%", "75%", "50%", "25%"]
with open("benchmark_b1024.csv", "w") as fin:
    fin.write(",".join(header))
    fin.write("\n")
    for key, value in percentile_data.items():
        fin.write(f"{key},")
        fin.write(",".join(map(lambda x: f'{x:.2f}',value)))
        fin.write("\n")
